# Code domain — Qwen3-8B LoRA finetune (Unsloth)

Run top to bottom on a Colab GPU runtime (Runtime → Change runtime type → T4 GPU, or A100 on Colab Pro).

Writes `domains/code/results/{baseline,finetuned}.json` and `domains/code/adapters/final/`. Commit those back to the `domain/code` branch when done — see the last cell.

In [ ]:
# Unsloth's exact install command changes with Colab's CUDA/torch version —
# check https://github.com/unslothai/unsloth for the current recommended cell
# if this one fails.
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q evalplus datasets huggingface_hub pyyaml

In [ ]:
import os

REPO_URL = "<your-repo-url>"  # set this to your llm-finetune-lab clone URL

if not os.path.exists("llm-finetune-lab"):
    !git clone -b domain/code $REPO_URL
%cd llm-finetune-lab

## 1. Data prep

Download → exact/near-dup dedup → decontaminate against HumanEval/MBPP. Same scripts you can run locally without a GPU — see `docs/ARCHITECTURE.md`.

In [ ]:
!python -m src.data.download --config domains/code/data_config.yaml
!python -m src.data.dedup --config domains/code/data_config.yaml
!python -m src.data.decontaminate --config domains/code/data_config.yaml

## 2. Baseline eval

Score the stock base model *before* touching the weights — this is what "finetuned" gets compared against. Runs against the base model repo id directly, no adapter.

In [ ]:
import yaml

base_cfg = yaml.safe_load(open("configs/base.yaml"))
base_model = base_cfg["base_model"]

!python -m src.eval.run_eval --model-path {base_model} --dataset humaneval --out domains/code/results/baseline.json
!python -m src.eval.run_eval --model-path {base_model} --dataset mbpp --out domains/code/results/baseline.json

## 3. Load model + attach LoRA

In [ ]:
from unsloth import FastLanguageModel

lora_cfg = base_cfg["lora"]
training_cfg = base_cfg["training"]

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_model,
    max_seq_length=base_cfg["max_seq_length"],
    load_in_4bit=base_cfg["load_in_4bit"],
)

model = FastLanguageModel.get_peft_model(
    model,
    r=lora_cfg["r"],
    lora_alpha=lora_cfg["alpha"],
    lora_dropout=lora_cfg["dropout"],
    target_modules=lora_cfg["target_modules"],
    bias=lora_cfg["bias"],
    use_gradient_checkpointing="unsloth",
    random_state=training_cfg["seed"],
)

## 4. Format dataset

`domains/code/data_config.yaml` names the fields (`question`/`answer` for glaive-code-assistant-v3) — change the keys below if you swap in a different domain dataset with a different schema.

In [ ]:
import json
from datasets import Dataset

records = [json.loads(l) for l in open("domains/code/data/processed/train.jsonl")]

def to_chat(example):
    messages = [
        {"role": "user", "content": example["question"]},
        {"role": "assistant", "content": example["answer"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

dataset = Dataset.from_list(records).map(to_chat)
print(dataset)

## 5. Train

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=base_cfg["max_seq_length"],
    args=SFTConfig(
        per_device_train_batch_size=training_cfg["per_device_train_batch_size"],
        gradient_accumulation_steps=training_cfg["gradient_accumulation_steps"],
        num_train_epochs=training_cfg["num_train_epochs"],
        learning_rate=training_cfg["learning_rate"],
        lr_scheduler_type=training_cfg["lr_scheduler_type"],
        warmup_ratio=training_cfg["warmup_ratio"],
        weight_decay=training_cfg["weight_decay"],
        optim=training_cfg["optim"],
        seed=training_cfg["seed"],
        output_dir="domains/code/adapters/checkpoints",
        logging_steps=10,
        save_strategy="no",
    ),
)

trainer_stats = trainer.train()

In [ ]:
model.save_pretrained("domains/code/adapters/final")
tokenizer.save_pretrained("domains/code/adapters/final")

## 6. Finetuned eval

Same harness, same seed, same benchmark as the baseline cell above — the diff between the two is the real result.

In [ ]:
!python -m src.eval.run_eval --model-path {base_model} --adapter-path domains/code/adapters/final --dataset humaneval --out domains/code/results/finetuned.json
!python -m src.eval.run_eval --model-path {base_model} --adapter-path domains/code/adapters/final --dataset mbpp --out domains/code/results/finetuned.json

In [ ]:
!python -m src.eval.report --domain code

## 7. Commit results back

```bash
git add domains/code/results domains/code/adapters/final
git commit -m "code domain: baseline vs finetuned eval results"
git push
```

Then update `docs/RESULTS.md` and the domain status table in the root `README.md` on `main`.